## Event log analysis

In [ ]:
import pandas as pd

# Load survey data
df = pd.read_csv('../survey data/placeholder.csv')
df = df[["RecordedDate", "Kinderopvang", "Starttijd werk", "Event log", "ID"]]

kinder_df = df[df["Kinderopvang"] == "De Melkfabriek"].copy()

# Filter out NaN values in the "Event log" column
filtered_df = kinder_df[kinder_df["Event log"].notna()]

# Calculate number of traces and unique resources
num_rows = len(filtered_df)
print("Number of traces:", num_rows)

num_resources = filtered_df["ID"].nunique()
print("Number of unique resources:", num_resources)

In [ ]:
# Create the event log
import json
from datetime import timedelta

# Convert RecordedDate to date only (YYYY-MM-DD)
filtered_df['RecordedDate'] = pd.to_datetime(
    filtered_df['RecordedDate'],
    errors='coerce'
).dt.date

# Use row number as trace_id
filtered_df['trace_id'] = filtered_df.index + 1

def explode_event_log(row):
    try:
        events = json.loads(row['Event log'])
    except (json.JSONDecodeError, TypeError):
        return []

    # Sort events by internal order
    events.sort(key=lambda x: x.get('order', 0))

    # Parse the start time of the day
    start_time = pd.to_datetime(
        row['Starttijd werk'],
        format='%H:%M',
        errors='coerce'
    )

    if pd.isna(start_time):
        return []

    exploded_rows = []

    for event in events:
        # Get date only (YYYY-MM-DD)
        date_only = str(row['RecordedDate'])
        timestamp = f"{date_only} {start_time.strftime('%H:%M:%S')}"
        activity = event.get('activity')

        exploded_rows.append({
            "trace_id": row['trace_id'],
            "process": event.get('process', ''),
            "employee": row.get('ID', ''),
            "activity": activity,
            "timestamp": timestamp,
            "duration": event.get('duration', ''),
            "energy": event.get('energy', None)
        })

        # Advance time
        start_time += timedelta(minutes=event.get('duration', 0))

    return exploded_rows

exploded_data = []
for _, row in filtered_df.iterrows():
    exploded_data.extend(explode_event_log(row))

exploded_df = pd.DataFrame(exploded_data)
exploded_df.reset_index(drop=True, inplace=True)

exploded_df.head(20)


In [ ]:
# Calculate number of events and activities
# Count the total number of events
num_rows = len(exploded_df)
print("Number of events:", num_rows)

# Count the total number of unique activities
unique_activities = exploded_df["activity"].unique()
print("Number of unique activities:", len(unique_activities))


In [ ]:
import pandas as pd
import numpy as np

# Group by trace_id and count activities per trace
tasks_per_trace = exploded_df.groupby('trace_id').size()
# Calculate the average number of tasks
avg_tasks_per_trace = tasks_per_trace.mean()

print("Average number of tasks per trace:", avg_tasks_per_trace)

## Top 3 activities per pattern

In [ ]:
# CORRECT FIGURE FOR PAPER
import pandas as pd
import numpy as np
from scipy.stats import zscore
import seaborn as sns
import matplotlib.pyplot as plt


files = [
    '../survey data/placeholder1.csv',
    '../survey data/placeholder2.csv',
    '../survey data/placeholder3.csv'
]

org_names = ['placeholder1', 'placeholder2', 'placeholder3']
top_n = 3

# Pattern 2 thresholds
gamma = 0.50   # fraction of observed days
delta = 0.50   # fraction of employees

# Preprocess event logs

def preprocess_event_log(file_path, org_name):
    df = pd.read_csv(file_path)
    df = df[["RecordedDate", "Kinderopvang", "Starttijd werk", "Event log", "ID"]]
    org_df = df[df["Kinderopvang"] == org_name].copy()
    org_df = org_df[org_df["Event log"].notna()]

    org_df['RecordedDate'] = pd.to_datetime(org_df['RecordedDate'], errors='coerce').dt.date
    org_df['trace_id'] = org_df.index + 1

    exploded_data = []

    def explode_event_log(row):
        try:
            events = json.loads(row['Event log'])
        except (json.JSONDecodeError, TypeError):
            return []

        events.sort(key=lambda x: x.get('order', 0))
        start_time = pd.to_datetime(row['Starttijd werk'], format='%H:%M', errors='coerce')
        if pd.isna(start_time):
            return []

        rows = []
        for event in events:
            timestamp = pd.Timestamp.combine(
                pd.to_datetime(row['RecordedDate']),
                start_time.time()
            )
            rows.append({
                "trace_id": row['trace_id'],
                "employee": str(row.get('ID', '')).strip(),
                "activity": str(event.get('activity', '')).strip(),
                "timestamp": timestamp,
                "duration": event.get('duration', 0),
                "energy": event.get('energy', None)
            })
            start_time += timedelta(minutes=event.get('duration', 0))
        return rows

    for _, row in org_df.iterrows():
        exploded_data.extend(explode_event_log(row))

    print(f"[INFO] Loaded {len(exploded_data)} events for {org_name}")
    return pd.DataFrame(exploded_data)

event_logs = {
    org_name: preprocess_event_log(file, org_name)
    for file, org_name in zip(files, org_names)
}

# Compute patterns
all_activities = []

for org_name, df in event_logs.items():
    df = df.copy()
    df['energy'] = pd.to_numeric(df['energy'], errors='coerce')
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['workday'] = df['timestamp'].dt.date
    df['employee'] = df['employee'].astype(str).str.strip()
    df['activity'] = df['activity'].astype(str).str.strip()
    df['time_of_day'] = np.where(df['timestamp'].dt.hour < 12, 'morning', 'afternoon')

    print(f"\n=== Processing organization: {org_name} ===")

    # Pattern 1: Low energy levels
    
    T = df['trace_id'].nunique()
    trace_counts = df.groupby('activity')['trace_id'].nunique().reset_index()
    trace_counts.columns = ['activity', 'n_traces']
    mean_energy = df.groupby('activity')['energy'].mean().reset_index()
    p1 = mean_energy.merge(trace_counts, on='activity')
    p1 = p1[p1['n_traces'] >= 0.40 * T]

    if len(p1) > 0:
        p1['severity'] = -p1['energy']
        p1['pattern'] = 'Pattern 1'
        p1['organization'] = org_name
        all_activities.append(p1[['activity','severity','pattern','organization']])

    # Pattern 2: Variation in energy

    days_per_employee = df.groupby('employee')['workday'].nunique().reset_index()
    days_per_employee.columns = ['employee', 'D_r']

    activity_days = df.groupby(['employee','activity'])['workday'].nunique().reset_index()
    activity_days.columns = ['employee','activity','days_activity']
    activity_days = activity_days.merge(days_per_employee, on='employee')
    activity_days = activity_days[activity_days['days_activity'] >= gamma * activity_days['D_r']]

    X = df['employee'].nunique()
    emp_median = df.groupby(['employee','activity'])['energy'].median().reset_index()
    emp_median = emp_median.merge(activity_days[['employee','activity']], on=['employee','activity'])

    def mad(x):
        x = x.dropna()
        return np.median(np.abs(x - np.median(x))) if len(x) > 1 else 0

    p2 = emp_median.groupby('activity').agg(
        n_employees=('employee','nunique'),
        variation=('energy', mad)
    ).reset_index()
    p2 = p2[p2['n_employees'] >= delta * X]

    if len(p2) > 0:
        p2['severity'] = p2['variation']
        p2['pattern'] = 'Pattern 2'
        p2['organization'] = org_name
        all_activities.append(p2[['activity','severity','pattern','organization']])
    
    # Pattern 3: Time-of-day effects 
    
    df['is_morning'] = (df['timestamp'].dt.hour < 12).astype(int)
    
    pa = df.groupby(['employee','activity','is_morning'])['energy'].mean().reset_index()
    
    pivot = pa.pivot_table(index=['employee','activity'], columns='is_morning', values='energy').reset_index()
    
    # Keep only employee-activity pairs with both morning and afternoon observations
    pivot = pivot.dropna(subset=[0,1])
    
    # Number of employees in organization
    num_employees = df['employee'].nunique()
    
    # Keep activities where at least 50% of employees have both morning & afternoon observations
    valid_counts = pivot.groupby('activity')['employee'].count().reset_index()
    valid_counts = valid_counts[valid_counts['employee'] >= 0.5 * num_employees]
    
    pivot = pivot[pivot['activity'].isin(valid_counts['activity'])]
    
    # Compute within-person energy difference: afternoon - morning
    pivot['delta'] = pivot[0] - pivot[1]
    
    # Aggregate per activity: severity is now simply the median difference magnitude
    p3 = pivot.groupby('activity').agg(
        severity=('delta', lambda x: abs(x.median()))
    ).reset_index()
    
    p3['pattern'] = 'Pattern 3'
    p3['organization'] = org_name
    all_activities.append(p3[['activity','severity','pattern','organization']])

    # Pattern 4: Start-of-day trap
    df_trap = df.copy()
    df_trap['flag'] = False
    for trace_id, trace_df in df_trap.groupby('trace_id'):
        trace_df = trace_df.sort_values('timestamp')
        n = len(trace_df)
        n_early = max(1, int(np.ceil(0.1 * n)))
        mean_duration = trace_df['duration'].mean()
        early_idx = trace_df.iloc[:n_early].index
        early_df = trace_df.loc[early_idx]
        
        short_and_low_energy_idx = early_df[
            (early_df['duration'] < mean_duration) 
        
        ].index
        
        df_trap.loc[short_and_low_energy_idx, 'flag'] = True

    p4 = df_trap.groupby('activity')['flag'].mean().reset_index()
    if len(p4) > 0:
        p4['severity'] = p4['flag']
        p4['pattern'] = 'Pattern 4'
        p4['organization'] = org_name
        all_activities.append(p4[['activity','severity','pattern','organization']])

    # Pattern 5: Long energy-draining
    activity_stats = df.groupby(['employee','activity']).agg(
        median_duration=('duration','median'),
        median_energy=('energy','median')
    ).reset_index()

    q75 = activity_stats.groupby('employee')['median_duration'].quantile(0.75).reset_index()
    q75.columns = ['employee','Q75']
    activity_stats = activity_stats.merge(q75, on='employee')

    activity_stats['long_and_draining'] = (
        (activity_stats['median_duration'] >= activity_stats['Q75']) &
        (activity_stats['median_energy'] < 0)
    )

    p5 = activity_stats.groupby('activity')['long_and_draining'].sum().reset_index()
    if len(p5) > 0:
        p5['severity'] = p5['long_and_draining']
        p5['pattern'] = 'Pattern 5'
        p5['organization'] = org_name
        all_activities.append(p5[['activity','severity','pattern','organization']])

    # Pattern 6: Repeated low-energy (<0)

    low_energy_df = df[df['energy'] < 0]
    daily_counts = low_energy_df.groupby(['employee','workday','activity']).size().reset_index(name='count')
    repeated = daily_counts[daily_counts['count'] >= 2]

    if len(repeated) > 0:
        repeated['z'] = zscore(repeated['count'])
        p6 = repeated.groupby('activity')['z'].sum().reset_index()
        p6['severity'] = p6['z']
        p6['pattern'] = 'Pattern 6'
        p6['organization'] = org_name
        all_activities.append(p6[['activity','severity','pattern','organization']])

# Normalize & explain top activities if any
if all_activities:
    all_activities_df = pd.concat(all_activities, ignore_index=True)
    all_activities_df = all_activities_df[all_activities_df['severity'] != 0]

    all_activities_df['severity_norm'] = all_activities_df.groupby(
        ['pattern','organization']
    )['severity'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 1
    )

    top_activities_df = all_activities_df.groupby(
        ['pattern','organization']
    ).apply(
        lambda x: x.sort_values('severity_norm', ascending=False).head(top_n)
    ).reset_index(drop=True)

    for (pattern, org), group in top_activities_df.groupby(['pattern','organization']):
        print(f"\n[INFO] Top {top_n} activities for Pattern '{pattern}' in organization '{org}':")
        for idx, row in group.iterrows():
            print(f"  - Activity: {row['activity']}, Normalized severity: {row['severity_norm']:.3f}")

# Step 5: Plot top activities with English translations
    import seaborn as sns
    import matplotlib.pyplot as plt

    sns.set(style='whitegrid', font_scale=1.8)


    org_palette = {
        'O2': '#1f77b4', # blue
        'O1': '#ff7f0e', # orange
        'O3': '#2ca02c'  # green
    }

    # Dutch → English activity translations
    translation_dict = {
        'Schoonmaken gedurende de dag': 'Cleaning throughout the day',
        'Opruimen gedurende de dag': 'Tidying up throughout the day',
        'Klaarmaken flesvoeding': 'Preparing bottle feeding',
        'Administratie algemeen': 'General administration',
        'Overdracht tussen diensten': 'Handover between shifts',
        'Begeleiden van vrij spel t.b.v. ontwikkeling kind': 'Guiding free play for child development',
        'Luiers (of kleren) verschonen': 'Changing diapers (or clothes)',
        'Naar bed brengen en uit bed halen kinderen': 'Putting children to bed and taking them out',
        'Breng/ophaalmoment kinderen begeleiden': 'Accompanying drop-off/pick-up of children',
        'Helpen bij wc-bezoek kinderen': 'Assisting children with toilet visits',
        'Lunch, fruit eten met kinderen begeleiden': 'Supervising lunch/fruit with children',
        'Locatie openen en sluiten': 'Opening and closing the location',
        'Spelactiviteiten begeleiden': 'Supervising play activities',
        'Mails lezen, beantwoorden': 'Reading and responding to emails',
        'Bestellingen doen': 'Placing orders',
        'Klaarmaken groente, lunch, fruit': 'Preparing vegetables, lunch, and fruit',
        'Administratie ouderverzoeken': 'Administration of parent requests',
        'Bestellingen inruimen': 'Storing orders',
        'Verplichte formulieren en lijsten bijhouden': 'Maintaining mandatory forms and lists',
        'Hulp bij naar buiten gaan van kinderen': 'Helping children going outside',
        'Bijhouden van dagelijkse bijzonderheden kind' : 'Recording daily child notes',
        'Bedden verschonen' : 'Changing beds',
        'VE - Thema\'s voorbereiden en plannen' : 'Preparing theme days'
    }

    # Apply translations
    top_activities_df['activity'] = top_activities_df['activity'].map(
        lambda x: translation_dict.get(x, x)
    )
    
    org_label_map = {
        'placeholder1': 'O1',
        'placeholder2': 'O2',
        'placeholder3': 'O3'
    }
    
    top_activities_df['organization'] = top_activities_df['organization'].map(org_label_map)
    
    pattern_labels = {
        'Pattern 1': 'P1: Low Energy Levels',
        'Pattern 2': 'P2: Variation in Energy Levels',
        'Pattern 3': 'P3: Time-Of-Day Effects',
        'Pattern 4': 'P4: Start-Of-Day Trap',
        'Pattern 5': 'P5: Long Energy-Draining Tasks',
        'Pattern 6': 'P6: Repeated Low-Energy Tasks'
    }
    
    top_activities_df['pattern'] = top_activities_df['pattern'].map(
        lambda x: pattern_labels.get(x, x)
    )


    g = sns.FacetGrid(
        top_activities_df,
        col='pattern',
        col_wrap=3,
        sharex=False,
        height=6,
        aspect=1.2
    )

    def barplot_fixed(data, **kwargs):
        # Sort activities by severity
        order = data.sort_values('severity_norm')['activity']
        sns.barplot(
            data=data,
            x='severity_norm',
            y='activity',
            order=order,
            hue='organization',
            palette=org_palette,
            orient='h',
            dodge=True
        )

    g.map_dataframe(barplot_fixed)
    g.add_legend(
    title='Organization',
    bbox_to_anchor=(0.5, -0.01),
    loc='upper center',
    ncol=3
)
    g.set_axis_labels("Impact score", "Activity")
    g.set_titles("{col_name}")
    plt.subplots_adjust(hspace=0.4, wspace=0.4)

    # Save figure
    g.savefig("Top_Activities_by_Pattern.pdf", bbox_inches='tight')
    print("\n[INFO] Saved figure to Top_Activities_by_Pattern_corrected_english.pdf")
else:
    print("[INFO] No activities passed thresholds for any pattern. Skipping normalization, explanation, and plotting.")



## Simulate interventions on event logs

In [ ]:
# Simulate interventions for top 3 activities per pattern

import pandas as pd
import numpy as np

top_n = 3

# Select top 3 activities per pattern per organization

top_activities_df = (
    all_activities_df
    .sort_values("severity_norm", ascending=False)
    .groupby(["pattern", "organization"])
    .head(top_n)
)

top_activity_dict = {}

for (pattern, org), group in top_activities_df.groupby(["pattern", "organization"]):
    top_activity_dict.setdefault(pattern, {})
    top_activity_dict[pattern][org] = group["activity"].tolist()

def mean_energy(df):
    return df["energy"].mean() if not df.empty else np.nan

results = []

# Apply interventions

for org_name, df_original in event_logs.items():

    df_original = df_original.copy()
    df_original["energy"] = pd.to_numeric(df_original["energy"], errors="coerce")
    df_original["duration"] = pd.to_numeric(df_original["duration"], errors="coerce")
    df_original["timestamp"] = pd.to_datetime(df_original["timestamp"])
    df_original["workday"] = df_original["timestamp"].dt.date

    base_mean = mean_energy(df_original)

    for pattern in ["Pattern 1","Pattern 2","Pattern 3",
                    "Pattern 4","Pattern 5","Pattern 6"]:

        df = df_original.copy()
        top_activities = top_activity_dict.get(pattern, {}).get(org_name, [])

        # PATTERN 1 — Remove frequent negative activities
        if pattern == "Pattern 1":
            for act in top_activities:
                df = df[df["activity"] != act]

        # PATTERN 2 — Reassign drained employees
        elif pattern == "Pattern 2":
            for act in top_activities:

                events = df[df["activity"] == act]
                if events.empty:
                    continue

                emp_median = events.groupby("employee")["energy"].median()

                if len(emp_median) < 2:
                    continue

                threshold = emp_median.median()
                drained = emp_median[emp_median < threshold].index.tolist()
                energized = emp_median[emp_median >= threshold].index.tolist()

                if not drained or not energized:
                    continue

                drained_idx = events[events["employee"].isin(drained)].index
                df = df.drop(drained_idx)

                new_events = events.loc[drained_idx].copy()

                high_idx = 0
                for idx in new_events.index:
                    target = energized[high_idx % len(energized)]
                    new_events.at[idx, "employee"] = target
                    new_events.at[idx, "energy"] = emp_median[target]
                    high_idx += 1

                df = pd.concat([df, new_events], ignore_index=True)

        
        # PATTERN 3 — Remove worst shift
        elif pattern == "Pattern 3":

            df["time_of_day"] = np.where(
                df["timestamp"].dt.hour < 12, "morning", "afternoon"
            )

            for act in top_activities:

                sub = df[df["activity"] == act]
                if sub.empty:
                    continue

                shift_energy = sub.groupby("time_of_day")["energy"].mean()

                if len(shift_energy) < 2:
                    continue
            
                # compute median energy per shift
                shift_energy = sub.groupby("time_of_day")["energy"].median()
                
                # pick the worst shift (lowest median energy)
                worst_shift = shift_energy.idxmin()
                
                mask = (
                    (df["activity"] == act) &
                    (df["time_of_day"] == worst_shift) &
                    (df["energy"] < 0)
                )
                df = df[~mask]
                
                sub = df[df["activity"] == act]

                shift_stats = sub.groupby("time_of_day")["energy"].agg(["mean", "count"])
                

        
        # PATTERN 4 — Remove start-of-the-day trap events
        elif pattern == "Pattern 4":

            for act in top_activities:

                for trace_id, trace_df in df.groupby("trace_id"):

                    trace_df = trace_df.sort_values("timestamp")
                    n_early = max(1, int(np.ceil(0.1 * len(trace_df))))
                    early_idx = trace_df.iloc[:n_early].index

                    mean_duration = trace_df["duration"].mean()

                    trap_idx = trace_df.loc[early_idx]
                    trap_idx = trap_idx[
                        (trap_idx["activity"] == act) &
                        (trap_idx["duration"] < mean_duration)
                    ].index

                    df = df.drop(trap_idx)

        
        # PATTERN 5 — Split long draining tasks
        elif pattern == "Pattern 5":

            for act in top_activities:

                activity_stats = (
                    df.groupby(["employee","activity"])
                    .agg(
                        median_duration=("duration","median"),
                        median_energy=("energy","median")
                    )
                    .reset_index()
                )

                q75 = (
                    activity_stats.groupby("employee")["median_duration"]
                    .quantile(0.75)
                    .reset_index()
                )
                q75.columns = ["employee","Q75"]

                activity_stats = activity_stats.merge(q75,on="employee")

                long_draining = activity_stats[
                    (activity_stats["activity"] == act) &
                    (activity_stats["median_duration"] >= activity_stats["Q75"]) &
                    (activity_stats["median_energy"] < 0)
                ]

                for _, row in long_draining.iterrows():

                    mask = (
                        (df["employee"] == row["employee"]) &
                        (df["activity"] == row["activity"])
                    )

                    for idx in df[mask].index:

                        original = df.loc[idx].copy()
                        d = original["duration"]
                        e = original["energy"]

                        if pd.isna(d) or d <= 0:
                            continue

                        df.loc[idx,"duration"] = d/2
                        df.loc[idx,"energy"] = e/2

                        energizing = original.copy()
                        energizing["timestamp"] += pd.Timedelta(minutes=d/2)
                        energizing["duration"] = 5
                        energizing["activity"] = "Energizing Break"
                        energizing["energy"] = abs(e) + 1

                        second_half = original.copy()
                        second_half["timestamp"] += pd.Timedelta(minutes=d/2 + 5)
                        second_half["duration"] = d/2
                        second_half["energy"] = e/2

                        df = pd.concat(
                            [df,
                             pd.DataFrame([energizing]),
                             pd.DataFrame([second_half])],
                            ignore_index=True
                        )

        # PATTERN 6 — Merge repeated draining events
        elif pattern == "Pattern 6":

            for act in top_activities:

                low_energy = df[
                    (df["activity"] == act) &
                    (df["energy"] < 0)
                ]

                daily_counts = (
                    low_energy.groupby(["employee","workday"])
                    .size()
                    .reset_index(name="count")
                )

                repeated = daily_counts[daily_counts["count"] >= 2]

                for _, row in repeated.iterrows():

                    mask = (
                        (df["employee"] == row["employee"]) &
                        (df["workday"] == row["workday"]) &
                        (df["activity"] == act)
                    )

                    idx = df[mask].index

                    if len(idx) >= 2:
                        df.loc[idx[0],"energy"] = df.loc[idx,"energy"].mean()
                        df = df.drop(idx[1:])

        # Compute results after intervention
        new_mean = mean_energy(df)
        delta = new_mean - base_mean
        pct = (delta / abs(base_mean)) * 100 if base_mean != 0 else 0

        results.append({
            "Organization": org_name,
            "Pattern": pattern,
            "ΔEnergy": round(delta, 3),
            "% Improvement": round(pct, 2)
        })


# PRINT TABLE

results_df = pd.DataFrame(results)

print("\n==============================")
print("ENERGY IMPROVEMENT (TOP 3 ACTIVITIES PER PATTERN)")
print("==============================\n")

pivot_abs = results_df.pivot(index="Organization",
                             columns="Pattern",
                             values="ΔEnergy")

pivot_pct = results_df.pivot(index="Organization",
                             columns="Pattern",
                             values="% Improvement")

print("ΔENERGY")
print(pivot_abs)
print("\n% IMPROVEMENT")
print(pivot_pct)
